# Numpy 之三：数据类型（dtype）

---

**ndarray 最大的特点：一个数组里的所有元素必须是同一个类型。**
这是它"省内存、算得快"的根基，也是它和 Python list 最本质的区别。

list 可以混装：`[1, "abc", 3.14, True]` 完全合法。
ndarray 不行 —— 但只要类型统一，就能压成**连续的定长内存块**。

## 学习目标

1. 认识 4 大类型族：布尔、整数、浮点、复数
2. 理解"位宽"（`int8/int16/int32/int64`）意味着什么
3. 亲手制造一次**整数溢出**，明白为什么位宽不能乱选
4. 会查 `np.iinfo` / `np.finfo` 看取值范围


In [ ]:
import numpy as np


---
## 一、布尔类型 `bool`


In [ ]:
arr = np.array([1, 0, 2, -1], dtype='bool')
print(arr)
print('dtype =', arr.dtype)


**规则：非 0 一律是 `True`，只有 0 是 `False`。**

注意 **负数也是 `True`**（`-1` -> `True`），
因为判断标准是"是否等于 0"，不是"是否大于 0"。

`bool` 只占 **1 字节**（8 位，虽然实际只用 1 位）。


In [ ]:
print('itemsize =', arr.itemsize, '（字节）')


---
## 二、整数类型：位宽决定能装多大

Numpy 的整数类型在"类型名后面带数字"，这个数字就是**位宽（bit）**：

| 类型 | 位宽 | 取值范围 |
|---|---|---|
| `int8` | 8 | -2⁷ ~ 2⁷-1，即 -128 ~ 127 |
| `int16` | 16 | -2¹⁵ ~ 2¹⁵-1，即 -32768 ~ 32767 |
| `int32` | 32 | 约 ±21 亿 |
| `int64` | 64 | 约 ±9.2×10¹⁸ |
| `uint8` | 8 | **无符号**，0 ~ 255 |

规律：**带符号**时，范围是 `-2^(n-1) ~ 2^(n-1)-1`（正负各占一半，0 占一格）。


In [ ]:
arr = np.array([0, 2, 127, 3], dtype='int8')
print(arr)
print('dtype =', arr.dtype, '，itemsize =', arr.itemsize, '字节')


### 用 `np.iinfo` 查范围，比背数字靠谱


In [ ]:
for dt in ['int8', 'int16', 'int32', 'int64']:
    info = np.iinfo(dt)
    print(f'{dt:>6} : {info.min:>22} ~ {info.max}')

print()
print('uint8  :', np.iinfo('uint8').min, '~', np.iinfo('uint8').max)


### ⚠️ 亲手制造一次"整数溢出"

`int8` 最大是 127。那 `127 + 1` 等于多少？

答案是 **-128**。因为它像里程表一样"翻回去了"（wrap around）。

**这是 Numpy 一个非常危险的坑**：它**不会报错**，只会静默给你一个错的结果。


In [ ]:
a = np.array([127], dtype='int8')
print('127 + 1 (int8) =', a + 1, '  <- 翻到了最小值，不报错！')

b = np.array([127], dtype='int16')
print('127 + 1 (int16) =', b + 1, '  <- 换成 int16 就正常了')

# 对比：Python 原生 int 是"无限精度"的，永远不溢出
print('\nPython 原生 127 + 1 =', 127 + 1, '（原生 int 不会溢出）')


**教训**：

- 涉及金额、ID、大数累加时，**别用 `int8/int16`**，用 `int64` 或让 Numpy 自动推断。
- 只有在**明确知道数值范围**（比如像素值 0~255）时才用窄类型省内存。


---
## 三、浮点类型 `float`

浮点数用 **IEEE 754** 标准存储，一个数被切成三段：

```
float32（单精度，4 字节）： 1 位符号 | 8 位指数 | 23 位尾数
float64（双精度，8 字节）： 1 位符号 | 11 位指数 | 52 位尾数
```

- **指数位**决定"能表示多大"（数量级范围）
- **尾数位**决定"能精确到几位"

所以 `float64` 比 `float32` 更精确，代价是内存翻倍。


In [ ]:
arr = np.array([0.0, 2.0, 127.0, 3.0], dtype='float32')
print(arr)
print('dtype =', arr.dtype, '，itemsize =', arr.itemsize, '字节')


In [ ]:
for dt in ['float16', 'float32', 'float64']:
    info = np.finfo(dt)
    print(f'{dt:>8} : 范围 {info.min:.4e} ~ {info.max:.4e}，精度 {info.precision} 位十进制')


> **修正原笔记**：原稿写 `float32` 的范围是"即 -2^128 到 2^128-1"。
> 这个说法不对 —— **浮点数的范围不看 2 的整数次幂，而由"指数位"决定**。
> 准确值请用 `np.finfo('float32')` 查（约 ±3.4×10³⁸）。
> 记不住没关系，**会查**比**背下来**重要。


### 浮点的经典陷阱：`0.1 + 0.2 != 0.3`


In [ ]:
print('0.1 + 0.2 =', 0.1 + 0.2)
print('等于 0.3 吗？', 0.1 + 0.2 == 0.3)

# 正确比较方式：用 np.isclose / np.allclose，允许一个极小的误差
print('用 isclose 比较：', np.isclose(0.1 + 0.2, 0.3))


**原因**：`0.1` 和 `0.2` 在二进制里都是**无限循环小数**，存进定长内存必须截断，
加起来就有极小的误差。**所以浮点数永远不要用 `==` 判断相等**，
要用 `np.isclose`（单个）或 `np.allclose`（数组）。


---
## 四、复数类型 `complex`

复数由"实部 + 虚部"组成，Python 里写 `3+4j`（`j` 是虚数单位）。
工程里常用（信号处理、电路分析），日常数据分析很少用到。


In [ ]:
arr = np.array([0.0 + 0.0j, 2.0 + 1.0j, 127.0 + 3.0j, 3.0 + 4.0j], dtype='complex64')
print(arr)
print('dtype =', arr.dtype, '，itemsize =', arr.itemsize, '字节')
print('实部：', arr.real)
print('虚部：', arr.imag)


> **修正原笔记**：原稿把 `complex64` 的范围注释**照抄了 `float32`**。
> 实际上 `complex64` = **两个 `float32`**（实部一个、虚部一个），
> 所以它占 8 字节，范围和精度都按 `float32` 来算。


---
## 五、类型转换：`astype()`

用 `astype()` 可以在类型之间搬迁，**括号里写目标类型**。


In [ ]:
a = np.array([1.9, 2.1, 3.7, -1.8])
print('原始 float：', a)

# float -> int：小数部分【直接截断】，不是四舍五入！
print('转 int     ：', a.astype(int), '  <- 小数被砍掉，不是四舍五入')

# 想四舍五入，要先 np.round 再转
print('先 round 再转：', np.round(a).astype(int))

# int -> float
print('转 float64 ：', np.array([1, 2, 3]).astype('float64'))

# bool -> int：True 变 1，False 变 0
print('bool 转 int ：', np.array([True, False, True]).astype(int))

# 数字 -> 字符串
print('转 str     ：', np.array([1, 2, 3]).astype(str))


**注意 `astype` 是"截断"不是"四舍五入"** ——
`1.9` 会变成 `1`，`-1.8` 会变成 `-1`（朝 0 方向截断）。
需要四舍五入就先 `np.round()`。

另外：`astype()` **返回新数组**，原数组不变。


In [ ]:
print('原数组还是：', a)


---
## 六、类型自动推断：`np.array` 会自己选

不写 `dtype` 时，Numpy 会"往宽了靠"：


In [ ]:
print('全是整数   ->', np.array([1, 2, 3]).dtype)
print('混入浮点   ->', np.array([1, 2, 3.0]).dtype, '  <- 小数把整型拉成了浮点')
print('混入布尔   ->', np.array([1, 2, True]).dtype)
print('全是小数   ->', np.array([1.5, 2.5]).dtype)
print('超 int32  ->', np.array([2 ** 40]).dtype, '  <- 太大，自动升到 int64')


**规律**：类型会"向上兼容" —— 整型遇到浮点变浮点，`int32` 装不下就升 `int64`。
好处是不用操心，坏处是**结果可能不是你想要的精度**，所以关键位置建议显式写 `dtype`。


---
## 七、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| `127 + 1` 用 `int8` | 变成 `-128`，**不报错** | 用 `int64`，或显式检查范围 |
| 以为 `float32` 范围是 2¹²⁸ | 说法不严谨 | 用 `np.finfo` 查实际范围 |
| `0.1 + 0.2 == 0.3` | `False` | 用 `np.isclose` / `np.allclose` |
| `astype(int)` 当四舍五入 | `1.9 -> 1` | 先 `np.round()` 再转 |
| 以为 `astype` 改了原数组 | 原数组没变 | 它**返回新数组** |
| 数组里塞混合类型 | 全被"拉宽"成同一种 | 这是设计使然；要混装就用 Python list |
| 字符串数字直接算术 | 报错或结果诡异 | 先 `astype(float)` |

---

## 八、动手练习

**练习 1**：创建一个 `int8` 数组装下 `[120, 121, 122]`，给它 `+10`，
观察结果，解释发生了什么。

**练习 2**：把 `[3.99, 0.5, -0.5, -3.99]` 转成 `int`，
说出每个元素的结果，并解释截断规则。

**练习 3**：用 `np.iinfo('uint8')` 查出范围，然后创建一个刚好在边界上的数组，
再加 1，看看会怎样。

**练习 4**：判断 `np.float16` 能精确表示到小数点后几位？
用 `np.finfo('float16').precision` 验证你的猜测。

**练习 5**：数组 `[1, 0, 5, -3, 0]` 转成 `bool`，结果是什么？为什么 `-3` 是 `True`？

---

**答案思路（先自己做）**

```python
# 1
a = np.array([120, 121, 122], dtype='int8')
print(a + 10)      # 溢出，翻到负数区间

# 2
print(np.array([3.99, 0.5, -0.5, -3.99]).astype(int))   # [3 0 0 -3]

# 3
print(np.iinfo('uint8').max)                # 255
u = np.array([255], dtype='uint8')
print(u + 1)                                # 0

# 5
print(np.array([1, 0, 5, -3, 0]).astype(bool))   # [True False True True False]
```
